<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/learn-ml-algorithms-2017-notebooks/blob/main/07-decision-trees.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/learn-ml-algorithms-2017/) collection._

# Chapter 7 — Decision Trees

Companion to [the chapter](https://www.ai.biz/books/learn-ml-algorithms-2017/decision-trees/).

A world of loan applications where the true rule is a set of yes-or-no questions, so a tree has something to find.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def make_loans(n, seed=0):
    """Income in thousands, debt-to-income ratio, missed payments last year, age. Did they default?"""
    rng = np.random.default_rng(seed)
    income = rng.uniform(15, 120, n)
    debt_ratio = rng.uniform(0.0, 0.8, n)
    missed = rng.integers(0, 4, n)
    age = rng.uniform(21, 70, n)                                    # recorded, and irrelevant
    risky = ((income < 40) & (debt_ratio > 0.4)) | (missed >= 2)   # the true rule
    default = np.where(risky, rng.uniform(size=n) < 0.7, rng.uniform(size=n) < 0.05).astype(int)
    return np.column_stack([income, debt_ratio, missed, age]), default

names = ['income', 'debt_ratio', 'missed', 'age']
X, y = make_loans(600, seed=1)
X_test, y_test = make_loans(5000, seed=2)
print(f'{len(y)} applications, {y.mean():.0%} defaulted')

## 1. How mixed is a group? Gini and entropy

In [ ]:
def gini(labels):
    p = np.mean(labels) if len(labels) else 0
    return 2 * p * (1 - p)

def entropy(labels):
    p = np.mean(labels)
    if p in (0, 1):
        return 0.0
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

for p in [0.0, 0.1, 0.3, 0.5]:
    labels = np.array([1] * int(p * 100) + [0] * (100 - int(p * 100)))
    print(f'{p:.0%} defaults: gini {gini(labels):.3f}  entropy {entropy(labels):.3f}')

## 2. Scoring one question

In [ ]:
def split_gain(X, y, feature, threshold):
    """How much less mixed the two halves are than the whole, weighted by their sizes."""
    left = X[:, feature] <= threshold
    if left.sum() == 0 or (~left).sum() == 0:
        return 0.0
    n = len(y)
    return gini(y) - (left.sum() / n) * gini(y[left]) - ((~left).sum() / n) * gini(y[~left])

print(f'gini of everyone: {gini(y):.3f}')
for feature, threshold in [(2, 1.5), (0, 40), (1, 0.4), (3, 45)]:
    print(f'  {names[feature]:<11} <= {threshold:<5} gain {split_gain(X, y, feature, threshold):.4f}')

## 3. The best question, by trying them all

In [ ]:
def best_split(X, y):
    best = (0.0, None, None)
    for feature in range(X.shape[1]):
        values = np.unique(X[:, feature])
        for threshold in (values[:-1] + values[1:]) / 2:           # midpoints between sorted values
            gain = split_gain(X, y, feature, threshold)
            if gain > best[0]:
                best = (gain, feature, threshold)
    return best

gain, feature, threshold = best_split(X, y)
print(f'best first question: {names[feature]} <= {threshold:.2f}   (gain {gain:.4f})')

## 4. Growing a tree, from scratch

In [ ]:
def grow(X, y, depth, max_depth, min_leaf=1):
    node = {'n': len(y), 'p': float(np.mean(y))}
    if depth == max_depth or len(y) < 2 * min_leaf or gini(y) == 0:
        return node
    gain, feature, threshold = best_split(X, y)
    if feature is None:
        return node
    left = X[:, feature] <= threshold
    if left.sum() < min_leaf or (~left).sum() < min_leaf:
        return node
    node.update(feature=feature, threshold=threshold,
                left=grow(X[left], y[left], depth + 1, max_depth, min_leaf),
                right=grow(X[~left], y[~left], depth + 1, max_depth, min_leaf))
    return node

def predict_proba(node, x):
    while 'feature' in node:
        node = node['left'] if x[node['feature']] <= node['threshold'] else node['right']
    return node['p']

def predict(tree, X):
    return np.array([predict_proba(tree, x) >= 0.5 for x in X]).astype(int)

def show(node, indent=''):
    if 'feature' not in node:
        print(f'{indent}-> default rate {node["p"]:.2f}  ({node["n"]} applicants)')
        return
    print(f'{indent}{names[node["feature"]]} <= {node["threshold"]:.2f}?')
    show(node['left'], indent + '  yes: ')
    show(node['right'], indent + '  no:  ')

tree = grow(X, y, 0, max_depth=3)
show(tree)
print(f'\naccuracy on new applications: {np.mean(predict(tree, X_test) == y_test):.3f}')

## 5. The one line you would ship

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

sk_tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)
print(export_text(sk_tree, feature_names=names, decimals=2))
print(f'scikit-learn, depth 3: {sk_tree.score(X_test, y_test):.3f}   ours: {np.mean(predict(tree, X_test) == y_test):.3f}')

## 6. Depth, and the tree that memorises

In [ ]:
print(f'{"depth":>5} {"leaves":>7} {"train":>7} {"test":>7}')
for depth in [1, 2, 3, 4, 6, 8, 12, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X, y)
    print(f'{str(depth):>5} {t.get_n_leaves():>7} {t.score(X, y):>7.3f} {t.score(X_test, y_test):>7.3f}')

In [ ]:
print(f'{"min_samples_leaf":>16} {"leaves":>7} {"train":>7} {"test":>7}')
for leaf in [1, 5, 20, 50]:
    t = DecisionTreeClassifier(min_samples_leaf=leaf, random_state=0).fit(X, y)
    print(f'{leaf:>16} {t.get_n_leaves():>7} {t.score(X, y):>7.3f} {t.score(X_test, y_test):>7.3f}')

## 7. A tree that predicts a number

In [ ]:
from sklearn.tree import DecisionTreeRegressor

def bicycle(km):
    return 10 + 3.2 * km + 0.15 * km**2

rng = np.random.default_rng(1)
km = rng.uniform(0.5, 8.0, 200)
minutes = bicycle(km) + rng.normal(0, 4.0, 200)
km_test = rng.uniform(0.5, 8.0, 1000)
min_test = bicycle(km_test) + rng.normal(0, 4.0, 1000)

for depth in [1, 2, 4, 8]:
    r = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(km.reshape(-1, 1), minutes)
    mae = np.mean(np.abs(r.predict(km_test.reshape(-1, 1)) - min_test))
    print(f'depth {depth}: {r.get_n_leaves():>3} leaves, {mae:.2f} min on new orders')
r = DecisionTreeRegressor(max_depth=4, random_state=0).fit(km.reshape(-1, 1), minutes)
print('\nbeyond the data:  12 km -> truth', round(bicycle(12), 1), ' tree', round(r.predict([[12]])[0], 1),
      '   16 km -> truth', round(bicycle(16), 1), ' tree', round(r.predict([[16]])[0], 1))

In [ ]:
grid = np.linspace(0.5, 8, 400).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(km, minutes, s=10, color='grey', alpha=0.6)
for depth, ls in [(2, '--'), (4, '-')]:
    r = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(km.reshape(-1, 1), minutes)
    ax.plot(grid, r.predict(grid), ls=ls, lw=2, label=f'depth {depth}')
ax.plot(grid, bicycle(grid), color='black', lw=1, label='truth')
ax.set_xlabel('distance (km)'); ax.set_ylabel('minutes'); ax.legend()
plt.show()

## 8. A staircase for a diagonal

In [ ]:
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(4)
A = rng.uniform(0, 1, (600, 2))
diag = (A[:, 0] + A[:, 1] > 1).astype(int)                 # the true boundary is a diagonal line
A_test = rng.uniform(0, 1, (5000, 2))
diag_test = (A_test[:, 0] + A_test[:, 1] > 1).astype(int)

print(f'logistic regression: {LogisticRegression().fit(A, diag).score(A_test, diag_test):.3f}')
for depth in [2, 4, 8, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(A, diag)
    print(f'tree, depth {str(depth):<4}: {t.score(A_test, diag_test):.3f}  ({t.get_n_leaves()} leaves)')

## 9. Small change in the data, different tree

In [ ]:
# Add a feature that says nearly the same thing as missed payments: the late fees charged last year
rng = np.random.default_rng(9)
late_fees = 12 * X[:, 2] + rng.normal(0, 1.5, len(y))
X9 = np.column_stack([X, late_fees])
names9 = names + ['late_fees']
X9_test = np.column_stack([X_test, 12 * X_test[:, 2] + rng.normal(0, 1.5, len(y_test))])

base = DecisionTreeClassifier(random_state=0).fit(X9, y)
roots, disagreements = [], []
for trial in range(20):
    sample = rng.choice(len(y), len(y), replace=True)              # the same 600 applications, resampled with replacement
    t = DecisionTreeClassifier(random_state=trial).fit(X9[sample], y[sample])
    roots.append(names9[t.tree_.feature[0]])
    disagreements.append(np.mean(t.predict(X9_test) != base.predict(X9_test)))
print('the root question, over 20 resamples:', {r: roots.count(r) for r in sorted(set(roots))})
print(f'fully grown trees disagree with the original on {np.mean(disagreements):.1%} of new applications')
print(f'the original tree is right on {base.score(X9_test, y_test):.1%} of them')

## 10. What the tree thinks matters

In [ ]:
full = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X, y)
for name, imp in sorted(zip(names, full.feature_importances_), key=lambda t: -t[1]):
    print(f'{name:<11} {imp:.3f}')

## Try it yourself

1. Replace `gini` with `entropy` in `split_gain`. Does the depth-3 tree change?
2. In `make_loans`, make the risky rule `income < 40 and debt_ratio > 0.4 and age < 30`. How deep does the tree need to be now, and does `age` appear in the importances?
3. In section 8, rotate the features so the boundary is axis-aligned: use `A[:, 0] + A[:, 1]` as a single new feature. How many leaves does the tree need now?